# 🗑️ Waste Classifier — Baseline CNN
### Notebook 03 | Model 1: Baseline CNN

A simple CNN with no augmentation, no BatchNorm, and no Dropout.
This serves as our reference point for comparing all future models.

**Architecture:** 3 Conv Blocks → Flatten → FC Layers → Softmax  
**Goal:** Establish a performance baseline before any improvements

## 📦 Imports & Setup

In [ ]:
import sys
sys.path.append(".")
from utils import *

import torch
import torch.nn as nn
import torch.optim as optim

set_seed(42)
device = get_device()
print("✅ Setup complete")

## 📁 Load Data

In [ ]:
DATA_DIR    = "../data/processed"
RESULTS_DIR = "../results"
MODELS_DIR  = "../models"

dataloaders, dataset_sizes, class_names = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=32,
    augment=False,              # ← No augmentation for baseline
    num_workers=0,              # set to 0 on Windows to avoid errors
    use_weighted_sampler=True,
)

print(f"Classes     : {class_names}")
print(f"Train size  : {dataset_sizes['train']}")
print(f"Val size    : {dataset_sizes['val']}")
print(f"Test size   : {dataset_sizes['test']}")

## 🏗️ Model Architecture — Baseline CNN

Three convolutional blocks followed by fully connected layers.
No BatchNorm, no Dropout, no augmentation — intentionally plain.

| Layer Block | Details |
|---|---|
| Conv Block 1 | Conv2d(3→32) + ReLU + MaxPool |
| Conv Block 2 | Conv2d(32→64) + ReLU + MaxPool |
| Conv Block 3 | Conv2d(64→128) + ReLU + MaxPool |
| FC 1 | 128×28×28 → 512 + ReLU |
| FC 2 | 512 → 5 (Softmax) |

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(BaselineCNN, self).__init__()

        # ── Conv Block 1 ──────────────────────────────
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 224 → 112
        )

        # ── Conv Block 2 ──────────────────────────────
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 112 → 56
        )

        # ── Conv Block 3 ──────────────────────────────
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 56 → 28
        )

        # ── Fully Connected ───────────────────────────
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x


model = BaselineCNN(num_classes=5).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## ⚙️ Training Configuration

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

print("Criterion : CrossEntropyLoss")
print("Optimizer : Adam (lr=0.001)")
print("Scheduler : ReduceLROnPlateau (factor=0.5, patience=3)")
print("Epochs    : 25")
print("Patience  : 7 (early stopping)")

## 🚀 Training

In [ ]:
history = train_model(
    model=model,
    dataloaders=dataloaders,
    dataset_sizes=dataset_sizes,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=25,
    patience=7,
    model_name="baseline_cnn",
    results_dir=RESULTS_DIR,
    models_dir=MODELS_DIR,
)

## 📈 Training Curves

In [ ]:
plot_history(
    history,
    model_name="baseline_cnn",
    results_dir=RESULTS_DIR,
)

## 🧪 Evaluation on Test Set

In [ ]:
# Load best checkpoint before evaluating
load_checkpoint(model, f"{MODELS_DIR}/baseline_cnn_best.pth", device)

# Test loss & accuracy
test_loss, test_acc = evaluate(model, dataloaders["test"], criterion, device)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")

## 📊 Classification Report & Confusion Matrix

In [ ]:
# Per-class metrics
report = get_classification_report(model, dataloaders["test"], device, class_names)

# Confusion matrix
cm = plot_confusion_matrix(
    model, dataloaders["test"], device, class_names,
    model_name="baseline_cnn",
    results_dir=RESULTS_DIR,
)

## 💾 Log Metrics

In [ ]:
log_metrics_to_csv("baseline_cnn", {
    "test_accuracy"   : round(test_acc, 4),
    "macro_f1"        : round(report["macro avg"]["f1-score"], 4),
    "macro_precision" : round(report["macro avg"]["precision"], 4),
    "macro_recall"    : round(report["macro avg"]["recall"], 4),
})

## ✅ Baseline CNN Summary

| Metric | Value |
|---|---|
| Architecture | 3 Conv Blocks + 2 FC Layers |
| Augmentation | None |
| BatchNorm | None |
| Dropout | None |
| Optimizer | Adam lr=0.001 |
| Scheduler | ReduceLROnPlateau |

> Results saved to `results/` and metrics logged to `results/all_models_metrics.csv`  


## 📂 Verify Model Checkpoint

In [ ]:
import os

checkpoint_name = "baseline_cnn_best.pth"
checkpoint_path = os.path.join(MODELS_DIR, checkpoint_name)

if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"✅ Checkpoint found: {checkpoint_path}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print(f"❌ Checkpoint NOT found: {checkpoint_path}")